In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import re
from pycontrails.core import GeoVectorDataset
from pycontrails.models.gpat.gpat import mc_test, boxm_test
from pycontrails.models.gpat.pp_gpat import GPATPostProcessor

In [ ]:
outputs_dir = f"{os.getcwd()}/outputs/"

In [ ]:
# Filter criteria
criteria = {
        # "n_ac": 3,
        # "rt_fl": (pd.Timedelta(minutes=30), pd.Timedelta(hours=2)),
        # "date_created": (pd.Timestamp("2024-11-16"), pd.Timestamp("2024-11-17")),
        "job_id": 'bg_run_NA_2022-01-01T12_00_00'
    }

In [ ]:
pp_gpat = GPATPostProcessor(outputs_dir, criteria)
pp_gpat.filtered_df


In [ ]:
# Create dicts to hold all necessary data (but no more)
fl_df_dict = {}
pl_df_dict = {}
chem_ds_dict = {}

In [ ]:
for job_id in pp_gpat.job_ids:
    # fl_df_dict[job_id] = pp_gpat.load_fl_df(job_id)
    # pl_df_dict[job_id] = pp_gpat.load_pl_df(job_id)
    chem_ds_dict[job_id] = pp_gpat.load_chem_ds(job_id, 11, 11, 2)



In [ ]:
# bg run plots
fig, ax = plt.subplots(2, 4, figsize=(20, 10))

for job_id, chem_ds in chem_ds_dict.items():
    
    print(chem_ds.variables)
    # get time var
    time = chem_ds["time"].values

    # get month from time var
    month = pd.to_datetime(time).month

    # species to plot
    NO = chem_ds["Y"].sel(species_out="NO").values
    NO2 = chem_ds["Y"].sel(species_out="NO2").values
    NOx = NO + NO2
    O3 = chem_ds["Y"].sel(species_out="O3").values
    OH = chem_ds["Y"].sel(species_out="OH").values
    HO2 = chem_ds["Y"].sel(species_out="HO2").values
    CO = chem_ds["Y"].sel(species_out="CO").values
    CH4 = chem_ds["Y"].sel(species_out="CH4").values

    NOy = pp_gpat.calc_NOy(chem_ds)
    NOz = pp_gpat.calc_NOz(chem_ds)

    NO2t = pp_gpat.calc_NO2t(chem_ds)
    O3_NOz = pp_gpat.calc_O3_NOz(chem_ds, NOz)
    HCHO_NO2 = pp_gpat.calc_HCHO_NO2(chem_ds)
    H2O2_HNO3 = pp_gpat.calc_H2O2_HNO3(chem_ds)
    alpha_CH3O2 = pp_gpat.calc_alpha_CH3O2(chem_ds)  



In [ ]:
ax[0, 0].plot(time, NO)
ax[0, 1].plot(time, NO2)
ax[0, 2].plot(time, O3)
ax[0, 3].plot(time, NOy)
ax[1, 0].plot(time, OH)
ax[1, 1].plot(time, HO2)
ax[1, 2].plot(time, CO)
ax[1, 3].plot(time, CH4)
# pp_gpat.plot_line_plot(ax[0, 1], time, NO2, 'NO2', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[0, 2], time, O3, 'O3', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[0, 3], time, NOy, 'NOy', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 0], time, OH, 'OH', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 1], time, HO2, 'HO2', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 2], time, CO, 'CO', 'Time', 'Concentration (ppb)')
# pp_gpat.plot_line_plot(ax[1, 3], time, CH4, 'CH4', 'Time', 'Concentration (ppb)')

fig.suptitle('Time series of species concentrations for bg run NA Jan')

plt.show()


In [ ]:
# chem_da = []
# emi_da = []
# # Create a figure and axis for plotting
# fig, ax = plt.subplots(figsize=(10, 6))

# for job, ds in enumerate(chem_ds):
#     chem_ds[job] = chem_ds[job].assign_coords(species_out=chem_ds[job].attrs["species_out"])
#     chem_da.append(chem_ds[job]["Y"])

#     emi_da.append(chem_ds[job]["emi"])

#     chem_da_cell = chem_da[job].sel(level=chem_da[0].level[1], longitude=-32.9, latitude=47.1, method='nearest')
#     emi_da_cell = emi_da[job].sel(level=emi_da[0].level[1], longitude=-32.9, latitude=47.1, method='nearest')

#     NO_data = chem_da_cell.sel(species_out="NO")
#     NO2_data = chem_da_cell.sel(species_out="NO2")
#     O3_data = chem_da_cell.sel(species_out="O3")

#     NO_emi_data = emi_da_cell.sel(emi_species="NO")
#     NO2_emi_data = emi_da_cell.sel(emi_species="NO2")

#     # Plot the time series data
#     #NO_data.plot(ax=ax, label=f"NO {chem_da_cell['job_id'].values[0]}")
#     NO_data.plot(ax=ax, label=f"NO Emissions {emi_da_cell['job_id'].values[0]}")
#     NO_emi_data.plot(ax=ax, label=f"NO Emissions {emi_da_cell['job_id'].values[0]}")

# # Add labels and legend
# ax.set_xlabel('Time')
# ax.set_ylabel('Concentration')
# ax.set_title('Time Series of Species Concentration at Selected Cell')
# ax.legend()

# # plt.show()
# pd.set_option('display.max_rows', 500)
# NO_df = NO_data.to_dataframe()
# NO_df

# NO_data.max()


In [ ]:

# chem_ds_stacked = chem_ds.stack(
#             {"cell": ["level", "longitude", "latitude"]}
#         )
# chem_ds_stacked = chem_ds_stacked.reset_index("cell")

# chem_ds_stacked
 

In [ ]:

# max_emi_cell = chem_ds_stacked["emi"].mean(dim="time").argmax()#.item()
# print(max_emi_cell)
# # find cell that has max emissions averaged over time in it
# cell_chem_ds = chem_ds_stacked.sel(job_id=job_ids[0], cell=max_emi_cell)
# cell_chem_ds
# # Select the emissions for the specified species
# emi_data = cell_chem_ds["emi"].sel(emi_species="NO")
# chem_data = cell_chem_ds["Y"].sel(species_out="NO")
# emi_data.plot()
# chem_data.plot()

# # # Convert time and emi data to pandas Series
# # ts = 0
# # time_series = pd.Series(emi_data["time"].values)
# # emi_series = pd.Series(emi_data.values)

# # # Print time and emi values side by side
# # for time, emi in zip(time_series, emi_series):
# #     ts += 1
# #     print(f"TS: {ts}, Time: {time}, EMI: {emi}")

# for s, species in enumerate(chem_ds["species"].values):
#     print(chem_ds["bg_chem"].isel(level=1,latitude=0,longitude=0, species=s).values)



In [ ]:
#anim_chem(job_ids[0], jobs_df, fl_df, pl_df, chem_ds[0], var1="emi", var2="NO", level=chem_da[0].level[1], resample_freq="2min")

In [ ]:
# vecmass, gridmass, mc = mc_test(job_ids[0], jobs_df, fl_df, pl_df, chem_ds)
# mc

In [ ]:
# path = '/user/work/kt16229/pycontrails_kt/pycontrails/models/gpat/'

# cell_chem_ds = boxm_test(path, job_ids[0], 0, chem_ds)

# # dj_data = cell_chem_ds["DJ"].sel(photol_coeffs=3)
# # dj_orig_data = cell_chem_ds["DJ_orig"].sel(photol_coeffs=3)
# # dj_data.plot()
# # dj_orig_data.plot()

# chem_data = cell_chem_ds["Y"].sel(species_out="NO")
# chem_orig_data = cell_chem_ds["Y_orig"].sel(species_out="NO")
# chem_data.plot()
# chem_orig_data.plot()
